# VectorCypher Retriever

The VectorCypher retriever enhances vector search by using a custom Cypher query to provide richer, more contextual answers from your knowledge graph.

**Learning Objectives:**
- Create a Cypher `retrieval_query` to enhance vector search results
- Use the `VectorCypherRetriever` class for contextual retrieval
- Combine semantic search with graph traversal

---

## Setup

Import required modules and initialize connections.

In [ ]:
import sys
sys.path.insert(0, '..')

from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

from config import get_neo4j_driver, get_llm, get_embedder

driver = get_neo4j_driver()
driver.verify_connectivity()

llm = get_llm()
embedder = get_embedder()

print(f"Connected to Neo4j!")
print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.model_id}")

## Custom Cypher Query

The VectorCypherRetriever executes your custom Cypher query for each chunk found by vector search.

The query receives the matched `node` (a Chunk) and can traverse the graph to gather additional context.

In [ ]:
# Custom Cypher query to get asset managers associated with companies
# Uses COLLECT subquery to limit asset managers per company
asset_manager_query = """
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(company:Company)
WITH node, company, COLLECT {
  MATCH (company)<-[:OWNS]-(manager:AssetManager)
  RETURN manager.managerName
  LIMIT 5
} AS managers
RETURN company.name AS company, managers AS AssetManagersWithSharesInCompany, node.text AS context
"""

print("Custom retrieval query defined!")

## Create VectorCypher Retriever

The `VectorCypherRetriever` combines:
1. **Vector search** - Finds relevant chunks using embeddings
2. **Graph traversal** - Applies your Cypher query to enrich each result

In [ ]:
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=asset_manager_query
)

print("VectorCypherRetriever initialized!")

## Query with Graph Context

Use the GraphRAG pipeline to answer questions that benefit from relationship context.

In [ ]:
query = "Who are the asset managers most affected by banking regulations?"

rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)
response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

## Inspect the Enriched Context

The context now includes both the text chunk AND the related asset managers from graph traversal.

In [ ]:
print("=== Enriched Context ===")
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[{i+1}] {item.content}")

## Finding Shared Risks Among Companies

Let's create a more complex query that finds risks shared between companies.

In [ ]:
# Query to find shared risk factors between companies
# Uses explicit grouping with WITH clause for modern Cypher
shared_risks_query = """
WITH node
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(c1:Company)
MATCH (c1)-[:FACES_RISK]->(risk:RiskFactor)<-[:FACES_RISK]-(c2:Company)
WHERE c1 <> c2
WITH c1, c2, risk
RETURN
  c1.name AS source_company,
  collect(DISTINCT c2.name)[0..10] AS related_companies,
  collect(DISTINCT risk.name)[0..10] AS shared_risks
LIMIT 10
"""

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    retrieval_query=shared_risks_query
)

print("Shared risks retriever created!")

In [ ]:
query = "What risks connect major tech companies?"

rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)
response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

In [ ]:
# View the context used in this query
print("=== Shared Risks Context ===")
for item in response.retriever_result.items:
    print(f"\n{item.content}")

## How VectorCypher Works

1. **Semantic Search** - Finds top-k text chunks most relevant to your query
2. **Graph Traversal** - For each chunk:
   - Follows relationships to connected entities
   - Discovers relationships between entities (e.g., shared risks)
   - Returns structured data alongside text context
3. **LLM Generation** - Uses both text and structured context to answer

**Why this is powerful:**
- Uses the chunk as a semantic anchor
- Discovers multi-entity relationships via graph logic
- Surfaces both context AND network insights

This approach is ideal for exploratory questions about relationships where you want structured, comparative insights.

## Summary

In this notebook, you learned:

1. **Custom Cypher queries** - Writing retrieval queries that traverse the graph
2. **VectorCypherRetriever** - Combining vector search with graph context
3. **Enriched context** - How graph data improves LLM answers
4. **Relationship discovery** - Finding connections between entities

The VectorCypher pattern is excellent when:
- Your question is about relationships or context
- You want to return structured entities alongside text
- Pure semantic search misses important graph connections

---

**Next:** [Text2Cypher Retriever](03_text2cypher_retriever.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")